In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader,Dataset

In [4]:
class ToxicCommentDataset(Dataset):
    def __init__(self,X,Y):
        self.x=torch.tensor(X, dtype=torch.long)
        self.y=torch.tensor(Y, dtype=torch.float32)

    def __len__(self):
        return len(self.x)
    
    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

In [5]:
class ToxicCommentMOdel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim,pad_idx):
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.fc= nn.Linear(hidden_dim, output_dim)
        self.sigmoid = nn.Sigmoid() if output_dim == 1 else nn.Softmax(dim=1)


    def forward(self, x):
        x=self.embedding(x)
        _, (h_n, _)=self.lstm(x)
        out =self.fc(h_n[-1])
        return self.sigmoid(out) 

In [6]:
## TRaining Setup
def train(model,dataloader,criterion,optimizer,device):
    model.train()
    total_loss = 0

    for X_batch,y_batch in dataloader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(dataloader)

In [7]:
## Evaluation Setup
def evaluate(model,dataloader,criterion,devices):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for X_batch,y_batch in dataloader:
            X_batch, y_batch = X_batch.to(devices), y_batch.to(devices)
            outputs = model(X_batch).squeeze()
            loss = criterion(outputs, y_batch)
            total_loss += loss.item()

    return total_loss / len(dataloader)

In [8]:
##  MAin Training Script
## Hyperparameters

vocab_size = 10000
embed_dim=128
hidden_dim=64
output_dim=1
pad_idx=0
batch_size=32
epochs=10
leaning_rate=0.001

# Devicce
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

##dataset
train_dataset=ToxicCommentDataset(X_train,y_train)
train_loader=DataLoader(train_dataset,batch_size=batch_size,shuffle=True)

## MOdel,Loss,Optimizer
model=ToxicCommentMOdel(vocab_size, embed_dim, hidden_dim, output_dim,pad_idx).to(device)
criterion=nn.BCEWithLogitsLoss() if output_dim == 1 else nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters(), lr=learning_rate)

## Training Loop
for epoch in range(epochs):
    loss= train(model, train_loader, criterion, optimizer, device)
    print(f'Epoch {epoch+1}/{epochs}, Loss: {loss:.4f}')

NameError: name 'X_train' is not defined

In [9]:
def predict(model,text_tensor):
    model.eval()
    with torch.no_grad():
        output=model(text_tensor.unsqueeze(0).to(device))
        return output.squeeze().cpu().numpy()